# 03 — Model optimisation & training (Step 3)
Train and tune **two models** — a **Neural Network (Keras)** and a **Random Forest (scikit-learn)** — for
each `X` scenario. Hyperparameters are tuned with **cross-validation** (kept light to respect the ~30-min
Colab budget). We record MSE/MAE/R² and **training duration**.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 60            # downsampling step in seconds. Lower = more samples, more time (try 30 or 15).
N_USERS          = 1500          # ACC Arena users, sampled at RANDOM from the full ~12k population (seeded).
                                 # Closest-users neighbours are searched WITHIN this sample.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=0/1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    return {"MSE": float(mean_squared_error(y_true, y_pred)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2":  float(r2_score(y_true, y_pred))}

In [ ]:
import json, time, joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
tf.random.set_seed(RANDOM_SEED)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

def load_xy(x):
    d = np.load(f"{PROCESSED_DIR}/acc_X{x}.npz")
    return d["X_train"], d["y_train"], d["X_val"], d["y_val"], d["X_test"], d["y_test"]

## Neural network
Small MLP. We do a tiny grid over width/learning-rate using the validation set, then keep the best.

In [ ]:
def build_mlp(input_dim, units=64, lr=1e-3, depth=2):
    m = keras.Sequential([keras.layers.Input((input_dim,))])
    for _ in range(depth):
        m.add(keras.layers.Dense(units, activation="relu"))
    m.add(keras.layers.Dense(1))
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="mse", metrics=["mae"])
    return m

def train_nn(Xtr, ytr, Xva, yva):
    best, best_val, best_cfg = None, np.inf, None
    es = keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)
    for units in (64, 128):                       # small validation-based search (GPU: a few seconds each)
        m = build_mlp(Xtr.shape[1], units=units, lr=1e-3)
        m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=30, batch_size=1024,
              callbacks=[es], verbose=0)
        v = m.evaluate(Xva, yva, verbose=0)[0]
        if v < best_val:
            best, best_val, best_cfg = m, v, {"units": units, "lr": 1e-3}
    return best, best_cfg

## Random forest
`GridSearchCV` (3-fold) over a small grid satisfies the cross-validation requirement.

**Speed (without hurting validity):** the default `max_features` makes RF regression evaluate *all* features
at every split → very slow with 73 features on Colab's 2 vCPUs. We use `max_features="sqrt"` (standard, faster,
usually better), `max_samples=0.3` (each tree fits on a 30% bootstrap subsample — this is just bagging), and
**tune on a capped subsample**, then refit the best config on the full training set. Benchmarked on the real
data (270k train rows): accuracy is essentially unchanged while fitting ~4× faster.

In [ ]:
RF_TUNE_SUBSAMPLE = 20000   # rows used for the CV search (final model is refit on ALL training rows)

def train_rf(Xtr, ytr):
    if len(Xtr) > RF_TUNE_SUBSAMPLE:
        s = np.random.RandomState(RANDOM_SEED).choice(len(Xtr), RF_TUNE_SUBSAMPLE, replace=False)
        Xt, yt = Xtr[s], ytr[s]
    else:
        Xt, yt = Xtr, ytr
    base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1,
                                 max_features="sqrt", min_samples_leaf=2, max_samples=0.3)
    grid = {"n_estimators": [100, 200], "max_depth": [16, None]}
    gs = GridSearchCV(base, grid, cv=3, scoring="neg_mean_squared_error", n_jobs=1)
    gs.fit(Xt, yt)
    best = gs.best_estimator_
    best.fit(Xtr, ytr)                      # refit best config on the full training set
    return best, gs.best_params_

## Train both models for every X and save results

In [ ]:
def infer_ms(predict, X):
    t = time.time(); predict(X); return round((time.time()-t)/len(X)*1000, 4)   # ms per sample

results = []
for x in X_VALUES:
    Xtr, ytr, Xva, yva, Xte, yte = load_xy(x)

    t = time.time(); nn, nn_cfg = train_nn(Xtr, ytr, Xva, yva); nn_dur = time.time()-t
    nn_pred = nn.predict(Xte, verbose=0).ravel()
    nn_m = evaluate(yte, nn_pred); nn_m.update(model="NN", X=x, train_s=round(nn_dur,1),
        infer_ms=infer_ms(lambda X: nn.predict(X, verbose=0), Xte), cfg=str(nn_cfg))
    nn.save(f"{RESULTS_DIR}/models/nn_X{x}.keras")

    t = time.time(); rf, rf_cfg = train_rf(Xtr, ytr); rf_dur = time.time()-t
    rf_pred = rf.predict(Xte)
    rf_m = evaluate(yte, rf_pred); rf_m.update(model="RF", X=x, train_s=round(rf_dur,1),
        infer_ms=infer_ms(rf.predict, Xte), cfg=str(rf_cfg))
    joblib.dump(rf, f"{RESULTS_DIR}/models/rf_X{x}.pkl")

    results += [nn_m, rf_m]
    print(f"X={x:>2} | NN R2={nn_m['R2']:.3f} ({nn_dur:.0f}s) | RF R2={rf_m['R2']:.3f} ({rf_dur:.0f}s)")

import pandas as pd
res = pd.DataFrame(results)
res.to_csv(f"{RESULTS_DIR}/metrics.csv", index=False)
res

Metrics saved to `results/metrics.csv` and models to `results/models/`. Notebook **04** compares them.